# OpenRouter model screening before Tinker training

One secret-carrying answer per APPS problem/model, at most **100 shared problems**.
Compare functional, exact-message, and joint pass@1. Inference is on OpenRouter;
later training would be on Tinker. This notebook does not train models.

**Preparation is free of inference calls.** It saves all requests and estimates cost.
The later execution cell asks yes/no before spending on OpenRouter and Modal.
See [README.md](README.md) for model availability, limitations, and artifact schemas.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

# Locate the repository when Jupyter starts in this notebook's subdirectory.
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / "ciphers/variable_naming_in_python_v2").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ciphers.variable_naming_in_python_v2.data.apps import REPO_ROOT
from ciphers.variable_naming_in_python_v2.data.codex_apps import SecretTask
from ciphers.variable_naming_in_python_v2.decoder import CipherConfig
from ciphers.variable_naming_in_python_v2.tinker.prepare import RunConfig, artifact_path, prepare_run
from ciphers.variable_naming_in_python_v2.tinker.evaluate import run_prepared, summarize

if not all(os.environ.get(key) for key in ("OPENROUTER_API_KEY", "STEGO_ARTIFACTS_DIR")):
    env_path = REPO_ROOT / ".env"
    if not env_path.is_file():
        raise FileNotFoundError("Set the environment variables or provide the repository-root .env")
    load_dotenv(env_path, override=False)
for key in ("OPENROUTER_API_KEY", "STEGO_ARTIFACTS_DIR"):
    if not os.environ.get(key):
        raise RuntimeError(f"Missing required environment variable: {key}")

In [ ]:
# Match the existing Codex notebook's cipher and payload.
secret = SecretTask(
    cipher=CipherConfig(
        special_variables={
            "sequence": ("values", "numbers"),
            "size": ("count", "size"),
            "accumulator": ("total", "sum_value"),
            "minimum": ("lowest", "minimum"),
            "maximum": ("highest", "maximum"),
            "result": ("answer", "result"),
        },
        length_bits=2,
    ),
    message_bits="101",
)
config = RunConfig(secret=secret, num_problems=100, max_tokens=16384, estimated_output_tokens=4096)
config

## Prepare and review

The next cell downloads the filtered APPS dataset and public model prices, writes all
requests, then costs the saved prompts. It sends **no inference requests**.
To review an existing run, replace the prepare call with `Path("variable_naming_v2/tinker/<run-id>")`.

The estimate assumes 4,096 output tokens per answer; the limit scenario assumes
16,384, including reasoning where the provider includes it in the limit.
Neither is a hard spending ceiling. Modal grading and account fees are excluded.

In [ ]:
run_dir = prepare_run(config)
print("Run directory:", artifact_path(run_dir))
estimate = json.loads((artifact_path(run_dir) / "estimate.json").read_text())
price_rows = []
for row in estimate["models"]:
    pricing = row["pricing"]
    price_rows.append({
        "model": row["model"], "available": row["available"], "requests": row["requests"],
        "input_characters": row["input_characters"], "estimated_input_tokens": row["input_tokens"],
        "input_USD_per_M": pricing["prompt"] * 1e6 if pricing else None,
        "output_USD_per_M": pricing["completion"] * 1e6 if pricing else None,
        "estimated_USD": row["estimated_usd"], "limit_scenario_USD": row["limit_scenario_usd"],
    })
display(pd.DataFrame(price_rows))
print(f"Estimated inference cost: ${estimate['estimated_usd']:.2f}")
print(f"Output-limit scenario: ${estimate['limit_scenario_usd']:.2f}")
print("Inspect requests.jsonl, grading_cases.json, and config.json in the run directory.")

## Optional paid execution

Run only after reviewing the saved requests and estimate. Answer `yes` to start;
anything else sends no inference requests. Execution is sequential, uses one answer
per problem/model, and saves results incrementally. An infrastructure error stops
the run. Started runs cannot be rerun automatically.

In [ ]:
answer = input(f"Run {sum(row['requests'] for row in estimate['models'])} saved requests? "
               f"Estimate ${estimate['estimated_usd']:.2f}; output-limit scenario "
               f"${estimate['limit_scenario_usd']:.2f}, plus Modal. Type yes/no: ").strip().lower()
if answer == "yes":
    results = run_prepared(run_dir, approved=True)
else:
    print("No inference requests sent.")

In [ ]:
# Safe to run before inference or after an interrupted run; incomplete rates stay blank.
display(pd.DataFrame(summarize(run_dir)))